<a href="https://colab.research.google.com/github/AgustinBiasca/Logistic-Regression/blob/main/Riesgo_crediticio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ISLP

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import seaborn as sns
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize)

from ISLP import confusion_table
from ISLP.models import contrast
from sklearn.discriminant_analysis import \
    (LinearDiscriminantAnalysis as LDA,
     QuadraticDiscriminantAnalysis as QDA)

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score


In [3]:
default = load_data('Default')
print(default.columns)
print(default.shape)

Index(['default', 'student', 'balance', 'income'], dtype='object')
(10000, 4)


En este data set tenemos 10.000 datos de personas con su respectivo ingreso, deuda, si son estudiantes o no y si defaultearon.

Vamos a buscar entrenar un modelo que nos permita predecir si una persona va a defaultear o no.

In [4]:
allvars = default.drop(columns=['default'])
design = MS(allvars)

X = design.fit_transform(default)
y = default.default == 'Yes'

glm = sm.GLM(y,
             X,
             family = sm.families.Binomial())

results = glm.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-10.869000,0.492000,-22.079,0.000
student[Yes],-0.646800,0.236000,-2.738,0.006
balance,0.005700,0.000000,24.737,0.000
income,0.000003,0.000008,0.370,0.712


intercept	-10.869000: Cuando todas las variables son 0. La probabilidad de default es altisima (logico)

balance 0.0057: Cuanto mayor es el balance mas probabilidad de defaultear

income 0.0000...3: Pareciera no ser significativo uno vez considerado el balance

El -0.646 de student[Yes] significa que si mantenemos fijo el balance y el ingreso, los estudiantes defaultean menos que los no estudiantes

**Usamos una regresion Binomial ya que es la distribucion que toman los datos. Un experimento con dos posibles resultado exito y fracaso.**

In [5]:
predict = results.predict()

predict

array([1.42872392e-03, 1.12220386e-03, 9.81227155e-03, ...,
       2.89617204e-03, 1.47143551e-01, 3.32282491e-05])

In [20]:
labels = pd.Series('No', index=default.index)
labels[predict > 0.05] = 'Yes'


In [18]:
confusion_table(labels, default.default)

Truth,No,Yes
Predicted,,
No,9390,130
Yes,277,203


En la diagonal principal tenemos los aciertos del modelo mientras que fuera de ella son errores

In [21]:
np.mean(labels == default.default)

np.float64(0.8983)

Nuestro modelo supuestamente tiene una precision del 89%.

Este numero, sin embargo, puede ser engañoso ya que al predecir con los mismos datos con los que entrenamos el modelo, esto puede generar que el modelo este memeorizando patrones en vez de aprenderlos. Vamos a solucionar esto y ver realmente que tan bueno es nuestro modelo.

In [22]:


# Separar en train y test antes de transformar
df_train, df_test = train_test_split(
    default, test_size=0.3, random_state=42, stratify=default['default']
)

# Crear variables objetivo (y)
y_train = (df_train['default'] == 'Yes').astype(int)
y_test  = (df_test['default']  == 'Yes').astype(int)

# Variables predictoras (X)
X_train_raw = df_train.drop(columns=['default'])
X_test_raw  = df_test.drop(columns=['default'])

# Aplicar el diseño correctamente
design = MS(X_train_raw)          # crear el objeto con las variables
X_train = design.fit_transform(df_train)  # ajustar solo con train
X_test  = design.transform(df_test)       # transformar test (sin fit)


Separamos el data set en conjuntos de entrenamiento y conjuntos de prueba. Tambien usamos respuesta de entrenamiento y respuesta de prueba

**test_size=0.3**

Indica qué fracción (o número absoluto) de los datos se reserva como conjunto de prueba. Aca 0.3 significa 30% test, 70% train.

**random_state=42**

Fija la semilla del generador aleatorio. Permite reproducibilidad: cada vez que ejecutemos el split con random_state=42 obtendrás la misma división. Si None, la división será diferente cada ejecución.

**stratify=y**

Asegura que la proporción de clases en y (por ejemplo %Yes/%No) se mantenga igual en y_train y y_test.

Muy importante cuando las clases están desbalanceadas (como en nuestro caso, donde los "Yes" son pocos). Sin stratify podríamos acabar con un test con muy pocos o ningún Yes.

fit_transform(df_train) → aprende la codificación/escala a partir del train.

transform(df_test) → aplica la misma transformación al test.

In [23]:
glm_train = sm.GLM(y_train,
             X_train,
             family = sm.families.Binomial())

results = glm_train.fit()
probs = results.predict(exog=X_test)

In [27]:
# Convertir probabilidades en etiquetas
pred_labels = np.where(probs >= 0.05, "Yes", "No")

print(pred_labels[:10])

['No' 'No' 'No' 'Yes' 'No' 'No' 'Yes' 'No' 'No' 'No']


In [28]:
y_test_labels = y_test.map({1: "Yes", 0: "No"})

confusion_table(pred_labels , y_test_labels )

Truth,No,Yes
Predicted,,
No,2610,13
Yes,290,87


In [29]:
np.mean(pred_labels == y_test_labels)

np.float64(0.899)

In [30]:
y_pred_bin = (probs >= 0.05).astype(int)

cm = confusion_matrix(y_test, y_pred_bin)
print(pd.DataFrame(cm, index=['Actual No','Actual Yes'], columns=['Pred No','Pred Yes']))
print(classification_report(y_test, y_pred_bin, target_names=['No','Yes']))
print("AUC:", roc_auc_score(y_test, probs))

            Pred No  Pred Yes
Actual No      2610       290
Actual Yes       13        87
              precision    recall  f1-score   support

          No       1.00      0.90      0.95      2900
         Yes       0.23      0.87      0.36       100

    accuracy                           0.90      3000
   macro avg       0.61      0.89      0.65      3000
weighted avg       0.97      0.90      0.93      3000

AUC: 0.9510689655172415


Se mantiene nuestro acierto del 89% en el modelo. Esto da que pensar, es muy poco probable que un modelo acierte en el 90% de los casos. Alguno de los problemas que pueden estar generando esto son:



*   Data Leaked (Fuga de datos): Esto puede suceder cuando una **variable predictora está directa o indirectamente causada por el evento que estamos tratando de predecir**. Este es uno de los principales problemas y hacen que nuestro modelo memorize en vez de aprender. A priori nuestro modelo es casi perfecto pero en la practica si tenemos Data Leaked cuando usemos el modelo y entren nuevos datos, la precision del modelo va a bajar drasticamente.
Por eso, siempre al construir un modelo hay que saber si las variables predictoras son independientes de la respuesta y sus datos conocidos previos a un posible default o no.
Otra causa de Data Leaked puede ser que el modelo para entrenar este usando informacion del conjunto de prueba. Esto ocurre cuando queremos normalizar los datos antes de dividir los conjuntos. Por eso en caso de ser necesario, hay que normalizar los conjuntos por separado.


* En este dataset, la cantidad de default 'no' es execesivamente alta. Por lo tanto cualquier modelo por mas malo que sea va a tener un gran numero de aciertos. Por esto, teoricamente es util para entender conceptos pero seria un grave error llevarlo a la practica.

